In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_68_try_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 69 ###

# Define question strings for clarity
question_of_interest_2019_cell81 = 'Have you ever used a TPU (tensor processing unit)?'
question_of_interest_2022_cell81 = 'Approximately how many times have you used a TPU (tensor processing unit)?'
# We'll work against the 2022 question text downstream
question_of_interest = question_of_interest_2022_cell81

# 1) Standardize/rename 2019 column and replace its values in one GPU‐friendly step
responses_df_2019_cell10 = responses_df_2019_cell10.rename(
    columns={question_of_interest_2019_cell81: question_of_interest}
)
responses_df_2019_cell10[question_of_interest] = (
    responses_df_2019_cell10[question_of_interest]
    .replace({
        '6-24 times': '6-25 times',
        '> 25 times': 'More than 25 times'
    })
)

# 2) Collect (df, year) pairs
df_list = [
    (responses_df_2019_cell10, '2019'),
    (responses_df_2020,       '2020'),
    (responses_df_2021,       '2021'),
    (responses_df_2022_cell10,'2022')
]

# 3) Concatenate and tag with year
combined = pd.concat(
    [df[[question_of_interest]].assign(year=year) for df, year in df_list],
    ignore_index=True
)

# 4) GPU counts by category & year
gpu_counts = (
    combined
      .groupby([question_of_interest, 'year'])
      .size()
      .reset_index(name='count')
)

# 5) Compute totals per year and merge
totals = (
    gpu_counts
      .groupby('year', as_index=False)['count']
      .sum()
      .rename(columns={'count': 'total'})
)
counts_with_totals = gpu_counts.merge(totals, on='year')

# 6) Calculate percentage & round
gpu_counts_final = counts_with_totals.assign(
    percentage=(counts_with_totals['count'] * 100 / counts_with_totals['total']).round(1)
)

# 7) Final formatting
x_axis_title_cell81 = 'Frequency of TPU usage'
df_combined_cell81 = (
    gpu_counts_final[[question_of_interest, 'percentage', 'year']]
      .rename(columns={question_of_interest: x_axis_title_cell81})
      .sort_values(by=['year', 'percentage'], ascending=True)
)

# Show result
df_combined_cell81.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 20 entries, 8 to 15
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Frequency of TPU usage  20 non-null     object
 1   percentage              20 non-null     float64
 2   year                    20 non-null     object
dtypes: float64(1), object(2)
memory usage: 752.0+ bytes
CPU times: user 376 ms, sys: 0 ns, total: 376 ms
Wall time: 374 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_69_try_2.pickle